In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ETL_FINAL").getOrCreate()

# Ruta base en Cloud Storage
bucket_path = "gs://bucket-ortiz-final/bronce/raw/"

# Leer los 3 datasets
df_flights = spark.read.option("header", True).option("inferSchema", True).csv(bucket_path + "Flight_delay.csv")

25/12/16 02:15:27 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:
df_flights.printSchema()


root
 |-- DayOfWeek: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- DepTime: integer (nullable = true)
 |-- ArrTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- UniqueCarrier: string (nullable = true)
 |-- Airline: string (nullable = true)
 |-- FlightNum: integer (nullable = true)
 |-- TailNum: string (nullable = true)
 |-- ActualElapsedTime: integer (nullable = true)
 |-- CRSElapsedTime: integer (nullable = true)
 |-- AirTime: integer (nullable = true)
 |-- ArrDelay: integer (nullable = true)
 |-- DepDelay: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Org_Airport: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Dest_Airport: string (nullable = true)
 |-- Distance: integer (nullable = true)
 |-- TaxiIn: integer (nullable = true)
 |-- TaxiOut: integer (nullable = true)
 |-- Cancelled: integer (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Diverted: integer (nullable = true

In [6]:
df_flights.show(5)

25/12/16 02:16:20 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+----------+-------+-------+----------+-------------+--------------------+---------+-------+-----------------+--------------+-------+--------+--------+------+--------------------+----+--------------------+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|DayOfWeek|      Date|DepTime|ArrTime|CRSArrTime|UniqueCarrier|             Airline|FlightNum|TailNum|ActualElapsedTime|CRSElapsedTime|AirTime|ArrDelay|DepDelay|Origin|         Org_Airport|Dest|        Dest_Airport|Distance|TaxiIn|TaxiOut|Cancelled|CancellationCode|Diverted|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+---------+----------+-------+-------+----------+-------------+--------------------+---------+-------+-----------------+--------------+-------+--------+--------+------+--------------------+----+--------------------+--------+------+-------+---------+----------------+--------+------------+------------+--------+----

Etapa 2

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ETL_FINAL").getOrCreate()


In [8]:
from pyspark.sql.functions import col

def limpiar_columnas(df):
    return df.select([col(c).alias(c.strip().replace("ï»¿", "").replace("\ufeff", "")) for c in df.columns])

In [9]:
bucket = "bucket-ortiz-final"
path_raw = f"gs://{bucket}/bronce/raw/"

# Vuelos
flights = spark.read.option("header", "true") \
    .option("encoding", "ISO-8859-1") \
    .csv(path_raw + "Flight_delay.csv")

flights_clean = limpiar_columnas(flights)


In [12]:
from pyspark.sql import functions as F
dim_date = (
    flights_clean
    .select(
        "Date",
        "DayOfWeek"
    )
    .withColumn("flight_date", F.to_date(F.col("Date"), "MM-dd-yyyy"))
    .dropDuplicates(["flight_date"])
)

In [13]:
dim_carrier = (
    flights_clean
    .select(
        "UniqueCarrier",
        "Airline"   # si no existe, bórrala
    )
    .dropDuplicates(["UniqueCarrier"])
)

In [14]:
from pyspark.sql import functions as F
dim_airport = (
    flights_clean
    .select(
        F.col("Origin").alias("airport_code"),
        F.col("Org_Airport").alias("airport_name")   # si no existe, bórrala
    )
    .unionByName(
        flights_clean.select(
            F.col("Dest").alias("airport_code"),
            F.col("Dest_Airport").alias("airport_name")  # si no existe, bórrala
        )
    )
    .dropDuplicates(["airport_code"])
)

In [15]:
dim_aircraft = (
    flights_clean
    .select("TailNum")
    .dropDuplicates(["TailNum"])
)

In [16]:

from pyspark.sql import functions as F
fact_flight_delay = (
    flights_clean
    .select(
        "Date",
        "UniqueCarrier",
        "Origin",
        "Dest",
        "TailNum",
        "FlightNum",
        "DepTime",
        "ArrTime",
        "CRSArrTime",
        "ActualElapsedTime",
        "CRSElapsedTime",
        "AirTime",
        "Distance",
        "TaxiIn",
        "TaxiOut",
        "ArrDelay",
        "DepDelay",
        "CarrierDelay",
        "WeatherDelay",
        "NASDelay",
        "SecurityDelay",
        "LateAircraftDelay",
        "Cancelled",
        "CancellationCode",
        "Diverted"
    )
    .withColumn("flight_date", F.to_date(F.col("Date"), "MM-dd-yyyy"))
    .withColumn("cant_vuelos", F.lit(1))
)

In [17]:
# Proyecto y dataset
bq_project = "examen-final-481401"
bq_dataset = "FinalOrtiz"

In [18]:
dim_date.write \
    .format("bigquery") \
    .option("table", f"{bq_project}.{bq_dataset}.dim_date") \
    .option("temporaryGcsBucket", "bucket-ortiz-final") \
    .mode("overwrite") \
    .save()

In [20]:
dim_carrier.write.format("bigquery").option("table", f"{bq_project}.{bq_dataset}.dim_carrier").option("temporaryGcsBucket", "bucket-ortiz-final").mode("overwrite").save()

dim_airport.write.format("bigquery").option("table", f"{bq_project}.{bq_dataset}.dim_airport").option("temporaryGcsBucket", "bucket-ortiz-final").mode("overwrite").save()

dim_aircraft.write.format("bigquery").option("table", f"{bq_project}.{bq_dataset}.dim_aircraft").option("temporaryGcsBucket", "bucket-ortiz-final").mode("overwrite").save()

fact_flight_delay.write.format("bigquery").option("table", f"{bq_project}.{bq_dataset}.fact_flights_delay").option("temporaryGcsBucket", "bucket-ortiz-final").mode("overwrite").save()